In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col, count, trim
import pyspark.sql.functions as f

In [0]:
df = spark.table('lakehouse.bronze.products')
df.display()

product_id,product_name,category,price
PROD-0001,Open Beaautiful,clo,32309.95
PROD-0002,Race Third,automotive,47346.82
PROD-0003,MEAN YOUR,3l3ctronics,-100
PROD-0004,Consumer Newspaper,toys,28506.95
PROD-0005,A Hot,clothing,19165.58
PROD-0006,How Over,clothing,2215.4
PROD-0007,Each Attorney,home,49101.1
PROD-0008,Idea Institution,clothing,20599.53
PROD-0009,Billion Animal,electronics,27710.51
PROD-0010,By Speaak,clothing_,0


## Data Quality Checks

#### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, f.StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

#### Consistency(Standardization)

In [0]:
df.select('product_name').distinct().show(10000, truncate=False)

+----------------------+
|product_name          |
+----------------------+
|Open Beaautiful       |
|Race Third            |
|MEAN YOUR             |
|Consumer Newspaper    |
|A Hot                 |
|How Over              |
|Each Attorney         |
|Idea Institution      |
|Billion Animal        |
|By Speaak             |
|Development How       |
|Bit Its               |
|Appear Position       |
|Ahead Once            |
|Mrs Audience          |
|StartWear             |
|Daughter Speech       |
|current agreement     |
|Section Drive         |
|StockYard             |
|Station Purpose       |
|Information Will      |
|Magazine Health6      |
|Nature While          |
|Allow Physical        |
|BringOffice           |
|Decade Detail         |
|Campaign Either       |
|Police Alone          |
|Business Still        |
|Century Old!          |
|HoldBefore            |
|Peace Magazine        |
|Southern Fall         |
|Level Tax             |
|Yard Participant      |
|Control Test          |


In [0]:
df = df.withColumn('product_name', f.regexp_replace(col('product_name'), '[\d!@*]', ''))\
       .withColumn('product_name', f.initcap(col('product_name')))

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/home/spark-a5ae82b5-4d8e-4686-94cb-c2/.ipykernel/336/command-8852935006684521-1845240969:1: SyntaxWarning: invalid escape sequence '\d'
  df = df.withColumn('product_name', f.regexp_replace(col('product_name'), '[\d!@*]', ''))\


In [0]:
df.select('category').distinct().show(100, truncate=False)

+------------+
|category    |
+------------+
|clo         |
|automotive  |
|3l3ctronics |
|toys        |
|clothing    |
|home        |
|electronics |
|clothing_   |
|sports      |
|hom3        |
|CLOTHING    |
|NULL        |
|beauty      |
|SPORTS      |
|aut         |
|TOYS        |
|kitchen     |
|bea         |
|hom         |
|HOME        |
|b3auty      |
|ELECTRONICS |
|home_       |
|kitch3n     |
|toy         |
|kitchen-    |
|automotive_ |
|kit         |
|KITCHEN     |
|BEAUTY      |
|toys-       |
|beauty_     |
|AUTOMOTIVE  |
|ele         |
|clothing-   |
|automotiv3  |
|home-       |
|sports-     |
|beauty-     |
|electronics_|
+------------+



In [0]:
df = df.withColumn('category', f.initcap(col('category')))\
       .withColumn('category',  f.regexp_replace(col('category'), '3', 'e'))\
       .withColumn('category', f.regexp_replace(col('category'), '[-_]', ''))

In [0]:
df = df.withColumn('category', when(col('category') == 'Kit', 'Kitchen')
                              .when(col('category') == 'Hom', 'Home')
                              .when(col('category') == 'Clo', 'Clothing')
                              .when(col('category') == 'Ele', 'Electronics')
                              .when(col('category') == 'Toy', 'Toys')
                              .when(col('category') == 'Aut', 'Automotive')
                              .when(col('category') == 'Bea', 'Beauty')
                              .otherwise(col('category')))

In [0]:
df.select('category').distinct().show(100, truncate=False)

+-----------+
|category   |
+-----------+
|Clothing   |
|Automotive |
|electronics|
|Toys       |
|Home       |
|Electronics|
|Sports     |
|NULL       |
|Beauty     |
|Kitchen    |
+-----------+



In [0]:
df.select('price').distinct().show(10000)

+---------+
|    price|
+---------+
| 32309.95|
| 47346.82|
|     -100|
| 28506.95|
| 19165.58|
|   2215.4|
|  49101.1|
| 20599.53|
| 27710.51|
|        0|
| 38495.28|
| 35426.08|
| 46067.97|
| 26010.64|
| 29803.79|
|    1,200|
|  30310.1|
|      NaN|
| 40642.04|
| 41128.19|
| 37907.67|
|  17246.0|
|  8088.09|
| 49986.32|
| 27304.07|
|   466.48|
| 25140.45|
| 41290.36|
| 15382.44|
| 25286.66|
|  34384.6|
|  7213.78|
| 42744.27|
|  4769.44|
|  9810.24|
| 47459.54|
| 34102.67|
| 36516.08|
| 37081.98|
| 19765.78|
| 30261.37|
| 39616.46|
| 46571.32|
| 12065.69|
|  7067.35|
| 18500.33|
| 16331.66|
| 49448.58|
| 15412.88|
| 30868.72|
| 38958.87|
| 48172.72|
| 16740.25|
|327124.57|
| 17559.72|
| 46482.66|
| 40476.32|
| 32660.43|
| 28109.74|
|  9950.52|
| 17910.72|
| 41450.04|
| 20912.84|
|  43838.0|
|  4459.16|
|  45637.6|
| 24604.88|
| 44789.91|
| 45881.34|
| 41058.72|
| 27265.71|
| 45707.56|
| 17261.44|
|  6277.16|
|   107.85|
| 125124.2|
|  2328.95|
| 48399.57|
|  1914.85|
| 44693.31|
| 30

In [0]:
df = df.withColumn('price', f.regexp_replace(col('price'), '[-,]', ''))\
       .withColumn('price', f.regexp_replace(col('price'), 'NaN', 0))

In [0]:
df = df.withColumn('price', f.round(col('price'), 2))

In [0]:
df.select('price').distinct().show(10000, truncate=False)

+---------+
|price    |
+---------+
|32309.95 |
|47346.82 |
|100.0    |
|28506.95 |
|19165.58 |
|2215.4   |
|49101.1  |
|20599.53 |
|27710.51 |
|0.0      |
|38495.28 |
|35426.08 |
|46067.97 |
|26010.64 |
|29803.79 |
|1200.0   |
|30310.1  |
|40642.04 |
|41128.19 |
|37907.67 |
|17246.0  |
|8088.09  |
|49986.32 |
|27304.07 |
|466.48   |
|25140.45 |
|41290.36 |
|15382.44 |
|25286.66 |
|34384.6  |
|7213.78  |
|42744.27 |
|4769.44  |
|9810.24  |
|47459.54 |
|34102.67 |
|36516.08 |
|37081.98 |
|19765.78 |
|30261.37 |
|39616.46 |
|46571.32 |
|12065.69 |
|7067.35  |
|18500.33 |
|16331.66 |
|49448.58 |
|15412.88 |
|30868.72 |
|38958.87 |
|48172.72 |
|16740.25 |
|327124.57|
|17559.72 |
|46482.66 |
|40476.32 |
|32660.43 |
|28109.74 |
|9950.52  |
|17910.72 |
|41450.04 |
|20912.84 |
|43838.0  |
|4459.16  |
|45637.6  |
|24604.88 |
|44789.91 |
|45881.34 |
|41058.72 |
|27265.71 |
|45707.56 |
|17261.44 |
|6277.16  |
|107.85   |
|125124.2 |
|2328.95  |
|48399.57 |
|1914.85  |
|44693.31 |
|30063.79 |
|423

#### Validity(converts data types)

In [0]:
df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



In [0]:
df = df.withColumn('price', col('price').cast('float'))

In [0]:
df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: float (nullable = true)



#### Uniqueness(Remove Duplicates)

In [0]:
df.groupBy('product_id').count().filter(col('count') > 1).show()

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



#### Completeness(Handling Nulls)

In [0]:
df.select([
    count(when(col(c).isNull(), True)).alias(f'{c}_Missing')
    for c in df.columns
]).show()

+------------------+--------------------+----------------+-------------+
|product_id_Missing|product_name_Missing|category_Missing|price_Missing|
+------------------+--------------------+----------------+-------------+
|                 0|                   0|              30|            0|
+------------------+--------------------+----------------+-------------+



In [0]:
df = df.withColumn('category', f.coalesce(col('category'), f.lit('No Info')))

In [0]:
df.select([
    count(when(col(c).isNull(), True)).alias(f'{c}_Missing')
    for c in df.columns
]).show()

+------------------+--------------------+----------------+-------------+
|product_id_Missing|product_name_Missing|category_Missing|price_Missing|
+------------------+--------------------+----------------+-------------+
|                 0|                   0|               0|            0|
+------------------+--------------------+----------------+-------------+



In [0]:
df.write.format('delta').mode('overwrite').saveAsTable('lakehouse.silver.products')